In [ ]:
# install required packages
%pip install langchain-ollama langchain-community rdflib

In [ ]:
# fix dependency issue by forcing a newer version of pyparsing
%pip install --upgrade pyparsing>=3.1.0

In [ ]:
from langchain_ollama import ChatOllama

# Create the LLM wrapper
ollama_llm = ChatOllama(
    base_url="https://ollama-gpt-oss.cluster.ai.wu.ac.at/",  # Ollama server URL
    model="gemma4:latest",                                    # name of the model you pulled
    temperature=1.0                                          # adjust temperature as needed              
)

In [ ]:
from langchain_community.graphs import RdfGraph
from langchain_community.chains.graph_qa.sparql import GraphSparqlQAChain
from IPython.display import Markdown, display
from langchain_core.prompts.prompt import PromptTemplate

# feeding the schema using a local RDF file
rdf_graph = RdfGraph(
    source_file="film.ttl",
    serialization="ttl",  # adjust if your file is xml/n3/etc.
    standard="owl",       # or "owl" if you’re using OWL
)
rdf_graph.load_schema()  # load the schema


# specify prompt for generating SPARQL SELECT query
SPARQL_GENERATION_SELECT_TEMPLATE = """
Task: Generate a SPARQL SELECT statement for querying a graph database.\n
For instance, to find all email addresses of John Doe, the following query in backticks would be suitable:\n```\nPREFIX foaf: <
  http://xmlns.com/foaf/0.1/
>\nSELECT ?email\nWHERE {{\n    ?person foaf:name "John Doe" .\n    ?person foaf:mbox ?email .\n}}\n```\n
Instructions:\nUse only the node types and properties provided in the schema.\nDo not use any node types and properties that are not explicitly provided.\n
Include all necessary prefixes.\nSchema:\n{schema}\nNote: Be as concise as possible.\nDo not include any explanations or apologies in your responses.\n
Generate only a valid SPARQL query, without any Markdown code block (triple backticks ```) around it.\n
Do not respond to any questions that ask for anything else than for you to construct a SPARQL query.\nDo not include any text except the SPARQL query generated.\n\nThe question is:\n{prompt}
"""

# specify prompt for generating SPARQL UPDATE query
SPARQL_GENERATION_UPDATE_TEMPLATE = """
Task: Generate a SPARQL UPDATE statement for updating a graph database.\n
For instance, to add \'jane.doe@foo.bar\' as a new email address for Jane Doe, the following query in backticks would be suitable:\n```\nPREFIX foaf: <
  http://xmlns.com/foaf/0.1/
>\nINSERT {{\n    ?person foaf:mbox <mailto:jane.doe@foo.bar> .\n}}\nWHERE {{\n    ?person foaf:name "Jane Doe" .\n}}\n```\n
Instructions:\nMake the query as short as possible and avoid adding unnecessary triples.\n
Use only the node types and properties provided in the schema.\nDo not use any node types and properties that are not explicitly provided.\n
Include all necessary prefixes.\nSchema:\n{schema}\nNote: Be as concise as possible.\nDo not include any explanations or apologies in your responses.\n
Generate only a valid SPARQL query, without any Markdown code block (triple backticks ```) around it.\n
Do not respond to any questions that ask for anything else than for you to construct a SPARQL query.\nReturn only the generated SPARQL query, nothing else.\n\nThe information to be inserted is:\n{prompt}
"""

# create the QA chain
chain = GraphSparqlQAChain.from_llm(
    allow_dangerous_requests=True,
    graph=rdf_graph,
    llm=ollama_llm,
    verbose=True,
    sparql_select_prompt=PromptTemplate(input_variables=['schema', 'prompt'], template=SPARQL_GENERATION_SELECT_TEMPLATE),
    sparql_update_prompt=PromptTemplate(input_variables=['schema', 'prompt'], template=SPARQL_GENERATION_UPDATE_TEMPLATE)
)

In [ ]:
# Query 1
result = chain.invoke({"query": "List all films and their properties."})
display(Markdown(result['result'])) 

In [ ]:
# Query 2
result = chain.invoke({"query": "Which films were directed by Julius Avery?"})
display(Markdown(result['result']))